In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_order_items = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_geolocation = pd.read_csv('olist_geolocation_dataset.csv')
df_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
datasets = {
    'customers': df_customers,
    'orders': df_orders,
    'products': df_products,
    'sellers': df_sellers,
    'order_items': df_order_items,
    'payments': df_order_payments,
    'reviews': df_order_reviews,
    'geolocation': df_geolocation,
    'translation': df_translation
}

for name, df in datasets.items():
    print(f'{name}:')
    print(df.columns.tolist())
    print()

In [ ]:
df_final = df_orders.merge(
    df_customers,
    on='customer_id',
    how='left'
)

df_final = df_final.merge(
    df_order_items,
    on='order_id',
    how='left'
)

df_final = df_final.merge(
    df_products,
    on='product_id',
    how='left'
)

df_final = df_final.merge(
    df_sellers,
    on='seller_id',
    how='left'
)

df_final = df_final.merge(
    df_translation,
    on='product_category_name',
    how='left'
)

In [ ]:
df_final.shape

1. Pedidos realizados entre vendedores e compradores geograficamente mais próximos apresentam menor tempo de entrega?

In [ ]:
#Ja que antes era possivel existir mais de um zip code prefix, ele agrupar pela media
#entao agora so existe uma localizacao aproximada para cada vendedor e comprador
geo = df_geolocation.groupby(
    'geolocation_zip_code_prefix'
).agg(
    latitude=('geolocation_lat', 'mean'),
    longitude=('geolocation_lng', 'mean')
).reset_index()

In [ ]:
df_final = df_final.merge(
    geo,
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)
df_final = df_final.rename(columns={
    'latitude': 'seller_lat',
    'longitude': 'seller_lng'
})

In [ ]:
df_final = df_final.drop(
    columns='geolocation_zip_code_prefix'
)

In [ ]:
df_final = df_final.merge(
    geo,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

In [ ]:
df_final = df_final.rename(columns={
    'latitude': 'customer_lat',
    'longitude': 'customer_lng'
})

In [ ]:
df_final = df_final.drop(
    columns='geolocation_zip_code_prefix'
)

In [ ]:
print(df_final)

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """
    A formula de Haversine calcula a distancia entre dois pontos tendo a 
    latitude e a longitude como parametro
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371 
    
    return c * r

In [ ]:
#Cria uma coluna chamada difference com a distancia entre o comprador e o vendedor 
#apos aplicar a formula de haversine
df_final['difference_km'] = haversine(
    df_final['customer_lng'],
    df_final['customer_lat'],
    df_final['seller_lng'],
    df_final['seller_lat']
)

In [ ]:
#transformando em datetime
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])


In [ ]:
#transformando em datetime
df_final['order_delivered_customer_date'] = pd.to_datetime(df_final['order_delivered_customer_date'])

In [ ]:
df_final['dias_ate_entrega'] = (df_final['order_delivered_customer_date'] 
- df_final['order_purchase_timestamp']).dt.total_seconds() / (60 * 60 * 24)

In [ ]:
print(df_final)

2. Categorias especificas de pedidos apresentam menor tempo de entrega?